# 03 Feature Engineering

This notebook validates the cleaning rules from `scratch.md`, builds the structured and NLP feature views, and exports a processed snapshot for inspection.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.pipeline import build_feature_views, build_text_projection, load_dataset, preprocess_dataframe
from src.settings import PROCESSED_DATA_DIR

bundle = load_dataset()
df, imputers = preprocess_dataframe(bundle.data)
_, _, tfidf_features = build_text_projection(df["feedback_clean"])
structured, with_sentiment, with_tfidf, with_all = build_feature_views(df, tfidf_features)

print("Duplicates removed:", bundle.duplicates_removed)
print("Processed rows:", len(df))

In [ ]:
df[["product_tier", "acquisition_channel", "nps_missing", "feedback_missing"]].head()

In [ ]:
print("Tier-specific imputers:")
imputers

In [ ]:
shape_summary = pd.DataFrame(
    {
        "view": ["structured", "structured_plus_sentiment", "structured_plus_tfidf", "all_nlp_features"],
        "rows": [len(structured), len(with_sentiment), len(with_tfidf), len(with_all)],
        "columns": [structured.shape[1], with_sentiment.shape[1], with_tfidf.shape[1], with_all.shape[1]],
    }
)
shape_summary

In [ ]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
export_cols = ["customer_id", "clv_12m", "log_clv_12m", "product_tier", "nps_category"]
df[export_cols].to_csv(PROCESSED_DATA_DIR / "modeling_snapshot.csv", index=False)
str(PROCESSED_DATA_DIR / "modeling_snapshot.csv")